# Required Imports

In [1]:
import os
import re
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
from numpy import dot
from numpy.linalg import norm
from collections import defaultdict
from scipy.ndimage import median_filter
from tqdm import tqdm

# -- IMPORTS FOR GRAPHS --
from torch_geometric.data import Data, Dataset, Batch
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, global_mean_pool, global_max_pool

## Parameters & Amplitude Sanitization Ablation Config (200 Packets)

In [2]:
# Configurazione PyG (200 pacchetti, stride=50, solo ampiezza)
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

device = torch.device("cpu")
print(f'device being used: {device}')

# Configuration settings
n_classes = 40
n_epochs = 100
train_batch_size = 32
test_batch_size = 1
learning_rate = 0.0005
num_node_features = 200  # 200 temporal packets per node (only amplitude)
num_nodes = 52

# Modalita' di filtraggio dell'ampiezza:
# 'none'   -> ampiezza caricata direttamente dal grafo (.pt)
# 'iqr'    -> clipping IQR per sottoportante lungo l'asse temporale
# 'hampel' -> filtro Hampel temporale (mediana scorrevole + MAD) per sottoportante
AMP_SANITIZATION_MODE = 'hampel'
HAMPEL_WINDOW = 7
HAMPEL_SIGMA = 3.0

data_root_base = './output_200p'
run_name = f'GNN_amp_{AMP_SANITIZATION_MODE}_200p_v2'
models_folder = os.path.join(data_root_base, 'models_amp_san_ablation_v2')

device being used: cpu


## Amplitude Filtering & Dataset Loaders (200 Packets)

In [3]:
def extract_id_from_filename(filename):
    match = re.search(r'(\d+[mf])', filename)
    return match.group(1) if match else None

def hampel_filter_temporal(signal_2d, window_size=7, n_sigmas=3.0):
    """
    Filtro Hampel vettorizzato lungo l'asse temporale (axis=1) per ciascuna delle 52 sottoportanti.
    Sostituisce gli outlier impulsivi locali con la mediana locale.
    """
    rolling_median = median_filter(signal_2d, size=(1, window_size), mode='nearest')
    abs_dev = np.abs(signal_2d - rolling_median)
    mad = median_filter(abs_dev, size=(1, window_size), mode='nearest')
    sigma_est = 1.4826 * mad
    outliers = abs_dev > (n_sigmas * sigma_est + 1e-8)
    filtered = np.where(outliers, rolling_median, signal_2d)
    return filtered.astype(np.float32)

def sanitize_amplitude_matrix(amp_np, mode='hampel', window_size=HAMPEL_WINDOW, n_sigmas=HAMPEL_SIGMA):
    """
    Filtraggio outlier per l'ampiezza (shape [N_subcarriers, T_packets], es. [52, 200]):
    - 'none': lascia invariato il tensore di ampiezza
    - 'iqr': clipping IQR per sottoportante lungo il tempo
    - 'hampel': filtro Hampel temporale su finestra scorrevole
    """
    if mode == 'none':
        return amp_np.astype(np.float32)
    elif mode == 'iqr':
        q25 = np.percentile(amp_np, 25, axis=1, keepdims=True)
        q75 = np.percentile(amp_np, 75, axis=1, keepdims=True)
        iqr = q75 - q25
        lower = q25 - 1.5 * iqr
        upper = q75 + 1.5 * iqr
        return np.clip(amp_np, lower, upper).astype(np.float32)
    elif mode == 'hampel':
        return hampel_filter_temporal(amp_np, window_size=window_size, n_sigmas=n_sigmas)
    else:
        raise ValueError(f'Modalita non supportata: {mode}')

def load_graph_with_amp_mode(path, mode='hampel'):
    data = torch.load(path, weights_only=False)
    if mode != 'none' and data.x is not None and data.x.dim() == 2:
        amp_np = data.x.detach().cpu().numpy().astype(np.float32)
        san_amp_np = sanitize_amplitude_matrix(amp_np, mode=mode)
        data.x = torch.from_numpy(san_amp_np).to(dtype=data.x.dtype, device=data.x.device)
    return data

class GraphReIDTrainDataset(Dataset):
    def __init__(self, data_root, mode='hampel'):
        super().__init__()
        self.data_root = data_root
        self.mode = mode
        self.samples, self.id_to_class = self._load_data()
        print(f"TRAIN: Trovati {len(self.samples)} grafi in {data_root} (amp_mode={self.mode})")

    def _load_data(self):
        paths = []
        for dirpath, _, filenames in os.walk(self.data_root):
            for f in filenames:
                if f.endswith('.pt'):
                    paths.append(os.path.join(dirpath, f))

        unique_ids = sorted(list(set([extract_id_from_filename(p) for p in paths if extract_id_from_filename(p)])))
        id_to_class = {orig_id: idx for idx, orig_id in enumerate(unique_ids)}

        samples = []
        for p in paths:
            pid = extract_id_from_filename(p)
            if pid:
                samples.append((p, id_to_class[pid]))
        return samples, id_to_class

    def len(self):
        return len(self.samples)

    def get(self, idx):
        path, target = self.samples[idx]
        data = load_graph_with_amp_mode(path, mode=self.mode)
        data.y = torch.tensor([target], dtype=torch.long)
        data.path = path
        return data

class PKSampler(torch.utils.data.Sampler):
    def __init__(self, dataset, P=8, K=4):
        self.dataset = dataset
        self.P = P
        self.K = K
        self.batch_size = self.P * self.K

        self.id_to_indices = defaultdict(list)
        for idx in range(len(self.dataset)):
            target = self.dataset.samples[idx][1]
            self.id_to_indices[target].append(idx)
        self.unique_ids = list(self.id_to_indices.keys())

    def __iter__(self):
        random.shuffle(self.unique_ids)
        batches = []
        for i in range(0, len(self.unique_ids), self.P):
            p_ids = self.unique_ids[i:i+self.P]
            if len(p_ids) < self.P:
                break
            batch = []
            for pid in p_ids:
                indices = self.id_to_indices[pid]
                if len(indices) >= self.K:
                    batch.extend(random.sample(indices, self.K))
                else:
                    batch.extend(np.random.choice(indices, size=self.K, replace=True))
            batches.append(batch)

        random.shuffle(batches)
        for batch in batches:
            yield from batch

    def __len__(self):
        return (len(self.unique_ids) // self.P) * self.batch_size

class GraphProbeGalleryValTest(Dataset):
    def __init__(self, data_root, mode='hampel'):
        super().__init__()
        self.data_root = data_root
        self.mode = mode
        self.data = self.read_folder()
        self.samples = self.create_samples()
        print(f"TEST: Trovati {len(self.data)} file totali. Generati {len(self.samples)} campioni Probe-Gallery (amp_mode={self.mode}).")

    def len(self):
        return len(self.samples)

    def get(self, idx):
        probe_pth, gallery_pths, is_match, probe_target, gallery_targets = self.samples[idx]
        probe_data = load_graph_with_amp_mode(probe_pth, mode=self.mode)
        gallery_data_list = [load_graph_with_amp_mode(item, mode=self.mode) for item in gallery_pths]
        return probe_data, gallery_data_list, is_match, probe_pth, gallery_pths, probe_target, gallery_targets

    def read_folder(self):
        paths = []
        for dirpath, _, filenames in os.walk(self.data_root):
            files = [f for f in filenames if f.endswith('.pt')]
            label = os.path.basename(os.path.normpath(dirpath))
            for item in files:
                paths.append([os.path.join(dirpath, item), label])
        return paths

    def create_samples(self):
        probes = [x[0] for x in self.data if x[1].lower() == 'probe']
        gallery = [x[0] for x in self.data if x[1].lower() == 'gallery']
        samples = []
        for probe in probes:
            probe_id = extract_id_from_filename(probe)
            if not probe_id:
                continue
            is_match, gallery_targets = [], []
            for item in gallery:
                item_id = extract_id_from_filename(item)
                if not item_id:
                    continue
                is_match.append(1 if probe_id == item_id else -1)
                gallery_targets.append(item_id)
            if len(gallery_targets) > 0:
                samples.append([probe, gallery, is_match, probe_id, gallery_targets])
        return samples

train_dataset = GraphReIDTrainDataset(os.path.join(data_root_base, 'GraphData', 'D1'), mode=AMP_SANITIZATION_MODE)
pk_sampler = PKSampler(train_dataset, P=8, K=4)
train_dataloader = DataLoader(train_dataset, batch_size=train_batch_size, sampler=pk_sampler, drop_last=True)

test_dataset = GraphProbeGalleryValTest(os.path.join(data_root_base, 'GraphData', 'D2t'), mode=AMP_SANITIZATION_MODE)
test_dataloader = DataLoader(test_dataset, batch_size=test_batch_size, shuffle=False)

TRAIN: Trovati 1360 grafi in ./output_200p/GraphData/D1 (amp_mode=hampel)
TEST: Trovati 1394 file totali. Generati 697 campioni Probe-Gallery (amp_mode=hampel).


# Graph Attention Network & Loss (Only Amplitude)

In [4]:
class GraphAttentionNetwork(nn.Module):
    def __init__(self, in_channels, temporal_dim, hidden_channels, out_channels, heads=4):
        super().__init__()
        self.in_channels = in_channels
        self.pos_emb = nn.Embedding(52, 16)

        self.temporal_conv = nn.Sequential(
            nn.Conv1d(in_channels=1, out_channels=16, kernel_size=5, padding=2),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )

        self.feature_fusion = nn.Sequential(
            nn.Linear(32 + 16, temporal_dim),
            nn.ReLU()
        )

        self.conv1 = GATConv(temporal_dim, hidden_channels, heads=heads, dropout=0.3)
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1, concat=False, dropout=0.3)

    def forward(self, x, edge_index, batch):
        N = x.size(0)
        subcarrier_idx = (torch.arange(N, device=x.device) % 52)
        pos_features = self.pos_emb(subcarrier_idx)

        # x ha forma [N, T] (solo ampiezza) -> [N, 1, T]
        x_seq = x.unsqueeze(1)
        temp_features = self.temporal_conv(x_seq).squeeze(-1)

        node_features = torch.cat([temp_features, pos_features], dim=1)
        node_features = self.feature_fusion(node_features)

        x_g, alpha1 = self.conv1(node_features, edge_index, return_attention_weights=True)
        x_g = F.elu(x_g)
        x_g, alpha2 = self.conv2(x_g, edge_index, return_attention_weights=True)

        x_mean = global_mean_pool(x_g, batch)
        x_max = global_max_pool(x_g, batch)

        global_features = torch.cat([x_mean, x_max], dim=1)
        global_features = F.normalize(global_features, p=2, dim=1)
        return global_features, alpha2

class GraphConsistencyTripletLoss(nn.Module):
    def __init__(self, margin=0.3, lambda_gc=0.005):
        super().__init__()
        self.ranking_loss = nn.MarginRankingLoss(margin=margin)
        self.lambda_gc = lambda_gc

    def forward(self, embeddings, attentions, targets, batch):
        n = embeddings.size(0)
        dist = torch.pow(embeddings, 2).sum(dim=1, keepdim=True).expand(n, n)
        dist = dist + dist.t()
        dist.addmm_(embeddings, embeddings.t(), beta=1, alpha=-2)
        dist = dist.clamp(min=1e-12).sqrt()

        mask = targets.expand(n, n).eq(targets.expand(n, n).t())
        dist_ap, dist_an = [], []
        for i in range(n):
            dist_ap.append(dist[i][mask[i]].max().unsqueeze(0))
            dist_an.append(dist[i][mask[i] == 0].min().unsqueeze(0))

        dist_ap = torch.cat(dist_ap)
        dist_an = torch.cat(dist_an)
        y = torch.ones_like(dist_an)
        triplet_loss = self.ranking_loss(dist_an, dist_ap, y)

        _, alpha = attentions
        gc_loss = torch.mean(alpha ** 2)
        return triplet_loss + (self.lambda_gc * gc_loss)

# Training and Evaluation Functions

In [5]:
def compute_ap(pos_ranks):
    if len(pos_ranks) == 0:
        return 0.0
    ap = 0.0
    for i, pos in enumerate(pos_ranks):
        ap += (i + 1) / (pos + 1)
    return ap / len(pos_ranks)

def testing_step(model, test_dataloader):
    model.eval()
    test_dataset = test_dataloader.dataset if hasattr(test_dataloader, 'dataset') else test_dataloader
    amp_mode = getattr(test_dataset, 'mode', AMP_SANITIZATION_MODE)

    if len(test_dataset.samples) == 0:
        print("\nWARNING: Test Dataset e' vuoto!")
        return 0.0, 0.0, 0.0, 0.0

    gallery_pths = test_dataset.samples[0][1]
    print(f"\nEstrazione feature della Gallery ({len(gallery_pths)} grafi)...")
    gallery_features_list, gallery_ids = [], []

    with torch.no_grad():
        batch_size = 32
        for i in range(0, len(gallery_pths), batch_size):
            batch_paths = gallery_pths[i:i+batch_size]
            batch_graphs = [load_graph_with_amp_mode(p, mode=amp_mode) for p in batch_paths]
            g_batch = Batch.from_data_list(batch_graphs).to(device)

            features, _ = model(g_batch.x, g_batch.edge_index, g_batch.batch)
            gallery_features_list.append(features.cpu())

            batch_ids = g_batch.subject_id
            if isinstance(batch_ids, str):
                batch_ids = [batch_ids]
            gallery_ids.extend(batch_ids)

        gallery_features = torch.cat(gallery_features_list, dim=0)

    print("\nValutazione Probes in corso...")
    samples = 0
    K = 10
    correct_at_k = np.zeros(K)
    AP_scores = []
    rank_matrix = []

    with torch.no_grad():
        for sample in tqdm(test_dataset.samples, leave=False):
            probe_pth = sample[0]
            probe_data = load_graph_with_amp_mode(probe_pth, mode=amp_mode)
            probe_batch = Batch.from_data_list([probe_data]).to(device)

            probe_feat, _ = model(probe_batch.x, probe_batch.edge_index, probe_batch.batch)
            probe_feat = probe_feat.cpu()

            similarities = F.cosine_similarity(probe_feat, gallery_features)
            scores = [[sim.item(), gal_id] for sim, gal_id in zip(similarities, gallery_ids)]

            samples += 1
            topk = sorted(scores, key=lambda element: (element[0]), reverse=True)
            p_id = probe_batch.subject_id[0] if isinstance(probe_batch.subject_id, list) else probe_batch.subject_id
            rank_matrix.append([p_id, topk])

            pos_ranks = [i for i, score in enumerate(topk) if p_id == score[1]]
            if pos_ranks:
                AP_scores.append(compute_ap(pos_ranks))

        for _, probe in enumerate(rank_matrix):
            p_id = probe[0]
            for i in range(K):
                if p_id == probe[1][i][1]:
                    correct_at_k[i:] += 1
                    break

    rank_accs = (correct_at_k / samples) * 100.0
    mAP = np.mean(AP_scores) if AP_scores else 0.0
    print(f'Total samples: {samples}, Rank #1: {rank_accs[0]:.3f}%, Rank #2: {rank_accs[1]:.3f}%, Rank #3: {rank_accs[2]:.3f}%, Rank #5: {rank_accs[4]:.3f}%, Rank #10: {rank_accs[9]:.3f}%, mAP:{mAP*100.0:.3f}%')
    return correct_at_k[0]/samples, correct_at_k[4]/samples, correct_at_k[9]/samples, mAP

def training_model(model, train_dataloader, test_dataloader, optimizer, criterion, n_epochs):
    bestRank1 = 0.0
    os.makedirs(models_folder, exist_ok=True)

    for epoch in range(n_epochs):
        model.train()
        train_loss = 0.0

        for batch_data in tqdm(train_dataloader, leave=False, total=len(train_dataloader)):
            batch_data = batch_data.to(device)
            targets = batch_data.y.view(-1)

            features, attentions = model(batch_data.x, batch_data.edge_index, batch_data.batch)
            loss = criterion(features, attentions, targets, batch_data.batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        rank1, _, _, _ = testing_step(model, test_dataloader)

        if rank1 > bestRank1:
            torch.save(model.state_dict(), os.path.join(models_folder, f'best_{run_name}_state_r1.bin'))
            bestRank1 = rank1

        print(f"Epoch: [{epoch+1}/{n_epochs}]| Loss: {train_loss/len(train_dataloader):.5f}")

## Run & Evaluate

In [6]:
model = GraphAttentionNetwork(in_channels=num_node_features, temporal_dim=128, hidden_channels=256, out_channels=256).to(device)
criterion = GraphConsistencyTripletLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

training_model(model, train_dataloader, test_dataloader, optimizer, criterion, n_epochs)

best_model = GraphAttentionNetwork(in_channels=num_node_features, temporal_dim=128, hidden_channels=256, out_channels=256).to(device)
best_model.load_state_dict(torch.load(os.path.join(models_folder, f'best_{run_name}_state_r1.bin'), map_location=device))
print('Model w/ best R1:')
testing_step(best_model, test_dataloader)


Estrazione feature della Gallery (697 grafi)...

Valutazione Probes in corso...


Total samples: 697, Rank #1: 51.506%, Rank #2: 54.663%, Rank #3: 57.532%, Rank #5: 61.836%, Rank #10: 69.727%, mAP:25.492%
Epoch: [1/100]| Loss: 0.32346



Estrazione feature della Gallery (697 grafi)...

Valutazione Probes in corso...


Total samples: 697, Rank #1: 56.098%, Rank #2: 59.110%, Rank #3: 61.980%, Rank #5: 65.854%, Rank #10: 72.023%, mAP:28.229%
Epoch: [2/100]| Loss: 0.31577



Estrazione feature della Gallery (697 grafi)...

Valutazione Probes in corso...


Total samples: 697, Rank #1: 61.263%, Rank #2: 66.858%, Rank #3: 69.154%, Rank #5: 72.597%, Rank #10: 77.188%, mAP:38.977%
Epoch: [3/100]| Loss: 0.30900


... (192 outputs left out)


Estrazione feature della Gallery (697 grafi)...

Valutazione Probes in corso...


Total samples: 697, Rank #1: 82.066%, Rank #2: 85.796%, Rank #3: 87.805%, Rank #5: 90.531%, Rank #10: 94.118%, mAP:70.238%
Epoch: [100/100]| Loss: 0.06527
Model w/ best R1:

Estrazione feature della Gallery (697 grafi)...

Valutazione Probes in corso...


Total samples: 697, Rank #1: 83.070%, Rank #2: 86.370%, Rank #3: 88.953%, Rank #5: 92.253%, Rank #10: 95.696%, mAP:70.498%


(np.float64(0.830703012912482),
 np.float64(0.9225251076040172),
 np.float64(0.9569583931133429),
 np.float64(0.7049766576369133))